# Characterization of the synthetic MDT scenario

**Purpose.** This notebook characterizes the data set that `task scenario` writes to
`data/raw/`. It examines the propagation environment, the network layout, and each stage of
the generation of the minimization-of-drive-tests (MDT) records. It also verifies the files
against the rules of the generator.

**Inputs.**

| File | Content |
|---|---|
| `scenario_config.yaml` | Configuration of the run |
| `cells.csv` | One row for each cell |
| `mdt.csv` | One row for each MDT record |

**Method.** Each section applies the functions of `src.simulation` and `src.scenario` again,
in the sequence of the generator, and compares the result with the files. All quantities come
from the three input files. The notebook contains no fixed numerical values, thus it remains
valid for a run with a different configuration.

**Figures.** `configure_matplotlib` sets the figure style. Each figure is saved as a PNG file
in `reports/figures/`. The text below each figure is its caption.

**Requirements.** Run `task scenario` first. The notebook needs an NVIDIA GPU because it
loads the Sionna RT scene.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
from omegaconf import OmegaConf
from scipy.constants import speed_of_light
from scipy.stats import truncnorm

from src.core import read_cells
from src.scenario.traffic import building_density, spatial_weights
from src.simulation import building_height_map, is_outdoor, load_rt_scene
from src.utils import (
    COLORS,
    DATA_DIR,
    FIGURE_WIDTH_DOUBLE,
    FIGURE_WIDTH_SINGLE,
    save_figure,
    setup_notebook,
)

RAW_DIR = DATA_DIR / "raw"
cfg = OmegaConf.load(RAW_DIR / "scenario_config.yaml")
device = setup_notebook(seed=cfg.seed)

cells = read_cells(RAW_DIR / "cells.csv")
cells_frame = pd.read_csv(RAW_DIR / "cells.csv")
mdt = pd.read_csv(RAW_DIR / "mdt.csv", parse_dates=["timestamp"])
rsrp_columns = [f"rsrp_{cell.name}" for cell in cells]
rsrp = mdt[rsrp_columns]

print(f"{len(cells)} cells, {len(mdt)} MDT records")

In [ ]:
def label_panels(axes):
    "Put the labels (a), (b), ... above the top left corner of each panel."
    for index, ax in enumerate(np.ravel(axes)):
        ax.set_title(f"({chr(ord('a') + index)})", loc="left", fontweight="bold")


def format_map(ax):
    "Give a map panel an equal scale on the two axes and axis labels in meters."
    ax.set(xlabel="$x$ (m)", ylabel="$y$ (m)", aspect="equal")
    ax.grid(False)

## 1. Configuration

The configuration below is that of the run which wrote the files.

In [ ]:
print(OmegaConf.to_yaml(cfg))

## 2. Propagation environment

### 2.1 Scene

`load_rt_scene` loads the Sionna RT scene. The scene supplies the geometry for the site
placement, for the UE positions, and for the ray tracer. Table 1 gives the objects of the
scene and their radio materials.

In [ ]:
scene = load_rt_scene(cfg.simulation.scene.name)
bbox = scene.mi_scene.bbox()
extent_m = np.array([bbox.max.x - bbox.min.x, bbox.max.y - bbox.min.y, bbox.max.z - bbox.min.z])

print(f"Scene: {cfg.simulation.scene.name}")
print(f"Extent (x, y, z): {extent_m[0]:.0f} m, {extent_m[1]:.0f} m, {extent_m[2]:.0f} m")
print(f"Area: {extent_m[0] * extent_m[1] / 1e6:.2f} km^2")
pd.DataFrame(
    [
        {"Object": name, "Radio material": scene_object.radio_material.name}
        for name, scene_object in scene.objects.items()
    ]
).style.set_caption("Table 1. Objects of the scene and their radio materials.")

### 2.2 Building height map

`building_height_map` casts one vertical ray at the center of each grid tile. The height
$h(\mathbf{x})$ is the elevation of the first surface above the ground. A grid tile is a
building if $h(\mathbf{x}) > h_{\min}$, where $h_{\min}$ is `grid.min_building_height_m`.
All other grid tiles are outdoor. Table 2 gives the statistics of the height map.

In [ ]:
height_map = building_height_map(scene, cfg.grid.tile_size_m)
heights = height_map.heights
ny, nx = heights.shape
size = height_map.tile_size
extent = [
    height_map.x_min,
    height_map.x_min + nx * size,
    height_map.y_min,
    height_map.y_min + ny * size,
]
is_building = heights > cfg.grid.min_building_height_m
outdoor = ~is_building
building_heights = heights[is_building]


def grid_index(x, y):
    "Return the row and the column of the grid tile that contains each position."
    col = np.clip(((np.asarray(x) - height_map.x_min) // size).astype(int), 0, nx - 1)
    row = np.clip(((np.asarray(y) - height_map.y_min) // size).astype(int), 0, ny - 1)
    return row, col


pd.Series(
    {
        "Grid tile size (m)": size,
        "Grid columns": nx,
        "Grid rows": ny,
        "Building threshold (m)": cfg.grid.min_building_height_m,
        "Building fraction of the area": is_building.mean(),
        "Outdoor fraction of the area": outdoor.mean(),
        "Median building height (m)": np.median(building_heights),
        "90th percentile of the building height (m)": np.percentile(building_heights, 90),
        "Maximum building height (m)": building_heights.max(),
        "Fraction of building tiles above the antenna height": (
            building_heights > cfg.sites.mast_height_m
        ).mean(),
    }
).round(3).to_frame("Value").style.set_caption("Table 2. Statistics of the height map.")

In [ ]:
fig, (ax_map, ax_hist) = plt.subplots(
    1, 2, figsize=(FIGURE_WIDTH_DOUBLE, 2.9), width_ratios=[1.5, 1]
)
# The scale stops at the 99th percentile: some towers are much higher than all
# other buildings and would compress the scale.
image = ax_map.imshow(
    heights, origin="lower", extent=extent, vmax=np.percentile(building_heights, 99)
)
fig.colorbar(image, ax=ax_map, label="Height above ground (m)", extend="max")
format_map(ax_map)

ax_hist.hist(building_heights, bins=50, color=COLORS[0])
ax_hist.axvline(cfg.sites.mast_height_m, color=COLORS[1], linestyle="--", label="Antenna height")
ax_hist.set(xlabel="Building height (m)", ylabel="Number of grid tiles")
ax_hist.legend()
label_panels([ax_map, ax_hist])
save_figure(fig, "scenario_height_map")
plt.show()

**Fig. 1.** Propagation environment. (a) Height of the highest surface above the ground.
The color scale stops at the 99th percentile of the building height.
(b) Distribution of the height of the building grid tiles. The dashed line is the antenna
height of the sites: a building above this line can obstruct a cell.

## 3. Network layout

`place_sites` puts the sites at the vertices of an equilateral triangle. The centroid is the
center of the scene plus `sites.center_offset_m`, and the side is the inter-site distance
`sites.isd_m`. A vertex in a building grid tile moves to the center of the nearest outdoor
grid tile. Each site has one cell for each azimuth in `sites.sector_azimuths_deg`. All
antennas are at `sites.mast_height_m` above the ground.

### 3.1 Cell table

Table 3 is the content of `cells.csv`.

In [ ]:
cells_frame.style.set_caption("Table 3. Content of cells.csv.").format(precision=2)

### 3.2 Site geometry

Table 4 compares each site with the configuration. Table 5 gives the distance between each
pair of sites. A distance that is not equal to `sites.isd_m` shows that a vertex moved out
of a building.

In [ ]:
sites = cells_frame.groupby("site")[["x", "y", "z"]].first()
row, col = grid_index(sites["x"], sites["y"])
sites["Surface height at the site (m)"] = heights[row, col]
sites["Antenna height above ground (m)"] = sites["z"] - height_map.z_ground
sites["Cells"] = cells_frame.groupby("site").size()

grid_x = height_map.x_min + (np.arange(nx) + 0.5) * size
grid_y = height_map.y_min + (np.arange(ny) + 0.5) * size
obstructed = []
for site in sites.itertuples():
    near = np.hypot(grid_x[None, :] - site.x, grid_y[:, None] - site.y) <= cfg.sites.isd_m / 2
    obstructed.append((heights[near] > site.z - height_map.z_ground).mean())
sites["Fraction of the area within ISD/2 above the antenna"] = obstructed
display(
    sites.style.set_caption("Table 4. Sites compared with the configuration.").format(precision=3)
)

xy = sites[["x", "y"]].to_numpy()
pairs = [(i, j) for i in range(len(xy)) for j in range(i + 1, len(xy))]
distances = pd.DataFrame(
    {
        "Sites": [f"{sites.index[i]} and {sites.index[j]}" for i, j in pairs],
        "Distance (m)": [np.linalg.norm(xy[i] - xy[j]) for i, j in pairs],
    }
)
distances["Difference from the ISD (m)"] = distances["Distance (m)"] - cfg.sites.isd_m
display(
    distances.style.set_caption("Table 5. Distances between the sites.")
    .format(precision=1)
    .hide(axis="index")
)

print(f"Configured antenna height: {cfg.sites.mast_height_m} m")
print(f"Azimuths in the file: {sorted(cells_frame['azimuth_deg'].unique().tolist())}")
print(f"Azimuths in the configuration: {list(cfg.sites.sector_azimuths_deg)}")
print(f"Downtilts in the file: {sorted(cells_frame['downtilt_deg'].unique().tolist())}")

In [ ]:
arrow_length = cfg.sites.isd_m / 5
site_marker = {"marker": "^", "color": "black", "edgecolor": "white", "s": 45, "zorder": 4}


def draw_sites(ax, annotate=True):
    "Draw each site and one arrow in the azimuth direction of each cell."
    for cell in cells:
        azimuth = np.deg2rad(cell.azimuth_deg)
        ax.annotate(
            "",
            xy=(cell.x + arrow_length * np.cos(azimuth), cell.y + arrow_length * np.sin(azimuth)),
            xytext=(cell.x, cell.y),
            arrowprops={"arrowstyle": "-|>", "color": "black", "linewidth": 1.0},
            zorder=3,
        )
    ax.scatter(sites["x"], sites["y"], **site_marker)
    if annotate:
        for index, site in sites.iterrows():
            ax.annotate(
                f"Site {index}",
                (site["x"], site["y"]),
                xytext=(5, 5),
                textcoords="offset points",
                fontweight="bold",
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.7, "pad": 1},
            )


fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_SINGLE * 1.3, FIGURE_WIDTH_SINGLE * 1.1))
ax.imshow(is_building, origin="lower", extent=extent, cmap="Greys", alpha=0.45)
draw_sites(ax)
format_map(ax)
save_figure(fig, "scenario_network_layout")
plt.show()

**Fig. 2.** Network layout. The gray areas are buildings. The triangles are the sites, and
each arrow is the azimuth of one cell.

### 3.3 Antenna and radio parameters

The simulation reads the transmit array of each cell from `cells.csv`. The receive array and
the solver parameters come from the key `simulation` of the configuration, which is
the content of `configs/simulation.yaml`. The bandwidth of the scene is one
resource element, thus `power_rs_dbm` is the transmit power in the RSRP calculation.
Table 6 gives the parameters. Each row that starts with "Solver" is one argument of the
Sionna RT `PathSolver`.

In [ ]:
array_columns = [
    "num_rows",
    "num_cols",
    "vertical_spacing",
    "horizontal_spacing",
    "pattern",
    "polarization",
    "power_rs_dbm",
]
transmit = cells_frame[array_columns].drop_duplicates().iloc[0]
receive = cfg.simulation.receiver
pd.Series(
    {
        "Carrier frequency (GHz)": cfg.simulation.radio.frequency_hz / 1e9,
        "Wavelength (cm)": round(speed_of_light / cfg.simulation.radio.frequency_hz * 100, 2),
        "Scene bandwidth = subcarrier spacing (kHz)": cfg.simulation.radio.subcarrier_spacing_hz
        / 1e3,
        "Reference-signal power for each resource element (dBm)": transmit["power_rs_dbm"],
        "Transmit array": f"{transmit['num_rows']} x {transmit['num_cols']}, "
        f"{transmit['pattern']}, {transmit['polarization']} polarization",
        "Transmit element spacing, vertical and horizontal (wavelengths)": (
            f"{transmit['vertical_spacing']}, {transmit['horizontal_spacing']}"
        ),
        "Receive array": f"{receive.num_rows} x {receive.num_cols}, {receive.pattern}, "
        f"{receive.polarization} polarization",
        **{f"Solver: {key}": value for key, value in cfg.simulation.solver.items()},
        "UE positions in one solver call": cfg.simulation.rx_batch_size,
    }
).to_frame("Value").style.set_caption("Table 6. Antenna and radio parameters.")

## 4. MDT records

### 4.1 Record table

Each row of `mdt.csv` is one MDT record. The records are statistically independent: the
generator does not simulate UE trajectories. An empty RSRP shows that the UE did not receive
the cell.

In [ ]:
display(mdt.head().style.set_caption("Table 7. First rows of mdt.csv.").format(precision=1))
pd.Series(
    {
        "Records": len(mdt),
        "First timestamp": mdt["timestamp"].min(),
        "Last timestamp": mdt["timestamp"].max(),
        "Distinct timestamps": mdt["timestamp"].nunique(),
        "RSRP columns": len(rsrp_columns),
    }
).to_frame("Value").style.set_caption("Table 8. Summary of mdt.csv.")

### 4.2 Temporal model

The number of records in slot $k$ is a Poisson random variable,

$$N_k \sim \mathrm{Poisson}(\mu_k), \qquad
\mu_k = N_{\mathrm{day}} \, \frac{a_{\mathrm{res}}(h_k) + a_{\mathrm{bus}}(h_k)}
{S \sum_{h} \left[ a_{\mathrm{res}}(h) + a_{\mathrm{bus}}(h) \right]},$$

where $N_{\mathrm{day}}$ is `traffic.records_per_day`, $h_k$ is the hour of slot $k$, $S$ is
the number of slots in one hour, and $a_{\mathrm{res}}$ and $a_{\mathrm{bus}}$ are the
residential and business profiles. The shapes of the two profiles agree with the residential
and office traffic patterns of Wang et al. (arXiv:1510.04026).

In [ ]:
residential = np.array(cfg.traffic.residential_profile)
business = np.array(cfg.traffic.business_profile)
hours_per_day = len(residential)
slots_per_hour = pd.Timedelta(hours=1) // pd.Timedelta(minutes=cfg.traffic.slot_minutes)
num_slots = cfg.traffic.num_days * hours_per_day * slots_per_hour

hourly = residential + business
expected = cfg.traffic.records_per_day * np.tile(
    np.repeat(hourly / hourly.sum(), slots_per_hour) / slots_per_hour, cfg.traffic.num_days
)
start = pd.Timestamp(cfg.traffic.start)
slot = ((mdt["timestamp"] - start) // pd.Timedelta(minutes=cfg.traffic.slot_minutes)).to_numpy()
observed = np.bincount(slot, minlength=num_slots)
hour = (slot // slots_per_hour) % hours_per_day
slot_time = np.arange(num_slots) / slots_per_hour

fig, (ax_profile, ax_slots) = plt.subplots(
    1, 2, figsize=(FIGURE_WIDTH_DOUBLE, 2.5), width_ratios=[1, 1.5]
)
ax_profile.step(range(hours_per_day), residential, where="mid", label="Residential")
ax_profile.step(range(hours_per_day), business, where="mid", linestyle="--", label="Business")
ax_profile.set(xlabel="Hour of day", ylabel="Relative traffic")
ax_profile.legend(loc="lower right", bbox_to_anchor=(1.0, 1.0), ncols=2)

ax_slots.bar(
    slot_time, observed, width=1 / slots_per_hour, align="edge", color=COLORS[5], label="Observed"
)
ax_slots.step(slot_time, expected, where="post", color=COLORS[1], label=r"Poisson mean $\mu_k$")
ax_slots.fill_between(
    slot_time,
    np.clip(expected - 2 * np.sqrt(expected), 0, None),
    expected + 2 * np.sqrt(expected),
    step="post",
    color=COLORS[1],
    alpha=0.2,
    linewidth=0,
    label=r"$\mu_k \pm 2\sqrt{\mu_k}$",
    zorder=0,
)
ax_slots.set(xlabel="Time since the start (h)", ylabel="Records in a slot")
ax_slots.xaxis.set_major_locator(MultipleLocator(hours_per_day / 4))
ax_slots.margins(x=0)
ax_slots.legend()
label_panels([ax_profile, ax_slots])
save_figure(fig, "scenario_temporal_model")
plt.show()

total_expected = expected.sum()
z_score = (len(mdt) - total_expected) / np.sqrt(total_expected)
inside = np.abs(observed - expected) <= 2 * np.sqrt(expected)
print(f"Expected number of records: {total_expected:.0f}. Observed: {len(mdt)}.")
print(f"Difference: {z_score:+.2f} standard deviations of the Poisson distribution.")
print(f"Fraction of slots in the band of two standard deviations: {inside.mean():.2f}")

**Fig. 3.** Temporal model. (a) Hourly traffic profiles of the configuration. (b) Number of
records in each slot, compared with the Poisson mean $\mu_k$ and with the band
$\mu_k \pm 2\sqrt{\mu_k}$.

### 4.3 Spatial model

`building_density` gives the building density $\rho(\mathbf{x})$: the fraction of building
grid tiles in a square window with the half side `traffic.density_radius_m`. For a record in
hour $h$, `spatial_weights` gives the probability of each outdoor grid tile,

$$p(\mathbf{x} \mid h) \propto
a_{\mathrm{res}}(h) \, \frac{\rho(\mathbf{x})}{\sum_{\mathbf{x}'} \rho(\mathbf{x}')}
+ a_{\mathrm{bus}}(h) \, \frac{\rho(\mathbf{x})^{\gamma}}{\sum_{\mathbf{x}'} \rho(\mathbf{x}')^{\gamma}},$$

where $\gamma$ is `traffic.business_exponent`. For $\gamma > 1$ the business component is
more concentrated in the dense areas than the residential component. The UE positions are
thus an inhomogeneous Poisson point process that changes during the day.

Fig. 4 shows $\rho$ and $p(\mathbf{x} \mid h)$ for the two hours at which the business
fraction of the traffic is at its maximum and at its minimum.

In [ ]:
density = building_density(height_map, cfg.grid.min_building_height_m, cfg.traffic.density_radius_m)
business_share = business / hourly
business_hour = int(np.argmax(business_share))
residential_hour = int(np.argmin(business_share))
weights = {
    h: spatial_weights(
        density,
        outdoor,
        h,
        cfg.traffic.residential_profile,
        cfg.traffic.business_profile,
        cfg.traffic.business_exponent,
    )
    for h in range(hours_per_day)
    if hourly[h] > 0
}
shown = [weights[business_hour], weights[residential_hour]]
vmax = max(weight.max() for weight in shown)

fig, axes = plt.subplots(1, 3, figsize=(FIGURE_WIDTH_DOUBLE, 2.6), sharey=True)
image = axes[0].imshow(np.where(outdoor, density, np.nan), origin="lower", extent=extent)
fig.colorbar(image, ax=axes[0], label=r"Building density $\rho$", location="top")
for ax, weight in zip(axes[1:], shown, strict=True):
    image = ax.imshow(
        np.where(outdoor, weight, np.nan), origin="lower", extent=extent, vmax=vmax, cmap="magma"
    )
bar = fig.colorbar(
    image, ax=axes[1:], label=r"Probability of a grid tile $p(\mathbf{x} \mid h)$", location="top"
)
bar.formatter.set_useMathText(True)
bar.update_ticks()
for ax in axes:
    format_map(ax)
for ax in axes[1:]:
    ax.set_ylabel("")
label_panels(axes)
save_figure(fig, "scenario_spatial_model")
plt.show()

entropy = {h: -(w[w > 0] * np.log(w[w > 0])).sum() for h, w in weights.items()}
for label, shown_hour in zip("bc", [business_hour, residential_hour], strict=True):
    print(
        f"Panel ({label}): {shown_hour}:00, business fraction {business_share[shown_hour]:.0%}, "
        f"entropy {entropy[shown_hour]:.3f} nat"
    )
print(f"Entropy of a uniform distribution on the outdoor tiles: {np.log(outdoor.sum()):.3f} nat")

**Fig. 4.** Spatial model. (a) Building density of the outdoor grid tiles. (b) Probability
map at the hour with the maximum business fraction. (c) Probability map at the hour with the
minimum business fraction. The text below the figure gives the two hours. A smaller entropy
shows a more concentrated map. Building grid tiles are white.

Fig. 5 shows the reported positions in `mdt.csv`. The records are in two groups: the hours
at which the business component is the larger component, and all other hours.

In [ ]:
row, col = grid_index(mdt["x"], mdt["y"])
mdt_density = density[row, col]
is_business_hour = business_share[hour] > 0.5
groups = {"Business hours": is_business_hour, "Residential hours": ~is_business_hour}

fig, axes = plt.subplots(1, 2, figsize=(FIGURE_WIDTH_DOUBLE, 3.1), sharey=True)
for ax, chosen in zip(axes, groups.values(), strict=True):
    image = ax.imshow(density, origin="lower", extent=extent, cmap="Greys", vmin=0, vmax=1)
    ax.scatter(mdt["x"][chosen], mdt["y"][chosen], s=3, color=COLORS[1], linewidths=0)
    draw_sites(ax, annotate=False)
    format_map(ax)
axes[1].set_ylabel("")
fig.colorbar(image, ax=axes, label=r"Building density $\rho$", shrink=0.8)
label_panels(axes)
save_figure(fig, "scenario_record_positions")
plt.show()

for label, (name, chosen) in zip("ab", groups.items(), strict=True):
    print(f"Panel ({label}): {name.lower()}, {chosen.sum()} records")

**Fig. 5.** Reported positions of the MDT records on the building density. (a) Hours at
which the business component is the larger component. (b) All other hours. The triangles are
the sites.

Table 9 is a quantitative test of the spatial model. For each group it compares the mean of
$\rho$ at the record positions with the expectation of the model,
$\mathrm{E}[\rho \mid h] = \sum_{\mathbf{x}} p(\mathbf{x} \mid h) \, \rho(\mathbf{x})$,
averaged over the hours of the records. The last row gives the mean of $\rho$ over all
outdoor grid tiles: this is the value for positions with a uniform distribution. The file
contains the reported positions, which include the GNSS error. A small difference between
the two columns is thus normal.

In [ ]:
model_mean = np.array(
    [(weights[h] * density).sum() if h in weights else np.nan for h in range(hours_per_day)]
)
comparison = pd.DataFrame(
    {
        "Records": {name: int(chosen.sum()) for name, chosen in groups.items()},
        "Mean density at the records": {
            name: mdt_density[chosen].mean() for name, chosen in groups.items()
        },
        "Expectation of the model": {
            name: model_mean[hour[chosen]].mean() for name, chosen in groups.items()
        },
    }
)
comparison.loc["All records"] = [len(mdt), mdt_density.mean(), model_mean[hour].mean()]
comparison.loc["Uniform outdoor positions"] = [np.nan, density[outdoor].mean(), np.nan]
comparison.style.set_caption(
    "Table 9. Building density at the record positions compared with the model."
).format({"Records": "{:.0f}"}, precision=3, na_rep="-")

### 4.4 UE position and height

The generator accepts a true position only if it is outdoor (`is_outdoor`). The reported
position is the true position plus a GNSS error,

$$\hat{x} = x + \varepsilon_x, \qquad \hat{y} = y + \varepsilon_y, \qquad
\varepsilon_x, \varepsilon_y \sim \mathcal{N}(0, \sigma_{\mathrm{GNSS}}^2),$$

where $\sigma_{\mathrm{GNSS}}$ is `measurement.gnss_std_m`. Some reported positions can thus
be in a building. The UE height above the ground has a normal distribution with the mean
`ue_height.mean_m` and the standard deviation `ue_height.std_m`, truncated to the interval
from `ue_height.min_m` to `ue_height.max_m`. Table 10 compares the file with this model.

In [ ]:
positions = mdt[["x", "y", "z"]].to_numpy()
reported_outdoor = is_outdoor(scene, positions)
ue_height = mdt["z"] - height_map.z_ground

low = (cfg.ue_height.min_m - cfg.ue_height.mean_m) / cfg.ue_height.std_m
high = (cfg.ue_height.max_m - cfg.ue_height.mean_m) / cfg.ue_height.std_m
height_model = truncnorm(low, high, loc=cfg.ue_height.mean_m, scale=cfg.ue_height.std_m)

display(
    pd.Series(
        {
            "GNSS standard deviation (m)": cfg.measurement.gnss_std_m,
            "Fraction of reported positions that are outdoor": reported_outdoor.mean(),
            "Fraction of reported positions in a building grid tile": is_building[row, col].mean(),
            "Mean UE height, file (m)": ue_height.mean(),
            "Mean UE height, model (m)": height_model.mean(),
            "Standard deviation of the UE height, file (m)": ue_height.std(),
            "Standard deviation of the UE height, model (m)": height_model.std(),
            "Minimum UE height, file (m)": ue_height.min(),
            "Maximum UE height, file (m)": ue_height.max(),
        }
    )
    .round(3)
    .to_frame("Value")
    .style.set_caption("Table 10. UE position and height.")
)

fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_SINGLE, 2.4))
ax.hist(ue_height, bins=30, density=True, color=COLORS[5], label="Observed")
grid = np.linspace(ue_height.min(), ue_height.max(), 300)
ax.plot(grid, height_model.pdf(grid), color=COLORS[1], label="Truncated normal model")
ax.set(xlabel="UE height above ground (m)", ylabel="Probability density (1/m)")
ax.legend()
save_figure(fig, "scenario_ue_height")
plt.show()

**Fig. 6.** Distribution of the UE height above the ground, compared with the truncated
normal distribution of the configuration.

### 4.5 RSRP

`compute_rsrp` finds the propagation paths from each cell $c$ to each UE position with the
Sionna RT path solver. The true RSRP is

$$\mathrm{RSRP}_c = P_{\mathrm{RS}} + 10 \log_{10} \sum_{n} \left| \mathbf{a}_{c,n}^{\mathsf{T}} \mathbf{p} \right|^2 \quad \text{(dBm)},$$

where $P_{\mathrm{RS}}$ is `power_rs_dbm`, $\mathbf{a}_{c,n}$ is the vector of the channel
coefficients of path $n$ for the transmit antennas, and $\mathbf{p}$ is the precoding vector
with equal weights. The sum is over the path powers: layer-3 filtering removes the fast
fading from a reported RSRP. The reported value is

$$\widehat{\mathrm{RSRP}}_c = \Delta \cdot \mathrm{round}\!\left(
\frac{\mathrm{RSRP}_c + w_c}{\Delta} \right), \qquad
w_c \sim \mathcal{N}(\mu_w, \sigma_w^2),$$

where $\mu_w$ is `measurement.rsrp_noise_mean_db`, $\sigma_w$ is
`measurement.rsrp_noise_std_db`, and $\Delta$ is `measurement.rsrp_resolution_db`. The record
does not contain a cell with $\widehat{\mathrm{RSRP}}_c$ below `measurement.min_rsrp_dbm`.
The generator samples a record again if no cell remains.

The file does not contain the true RSRP. This notebook can thus not show the measurement
error of each record.

In [ ]:
cells_per_record = rsrp.notna().sum(axis=1)
detection = rsrp.notna().mean().rename(lambda name: name.removeprefix("rsrp_"))
strongest_rsrp = rsrp.max(axis=1)
received = rsrp.to_numpy()[np.isfinite(rsrp.to_numpy())]
site_of_cell = cells_frame.set_index("cell")["site"]
site_order = list(sites.index)
sector_markers = ["o", "s", "D", "v", "P", "X"]
sector_of_cell = cells_frame.groupby("site").cumcount().set_axis(cells_frame["cell"])


def ecdf(values):
    "Return the sorted values and the empirical cumulative distribution function."
    ordered = np.sort(values)
    return ordered, np.arange(1, ordered.size + 1) / ordered.size


fig, (ax_detect, ax_count, ax_cdf) = plt.subplots(
    1, 3, figsize=(FIGURE_WIDTH_DOUBLE, 2.6), width_ratios=[1.2, 0.9, 1.2]
)
bar_colors = [COLORS[site_order.index(site_of_cell[name])] for name in detection.index]
ax_detect.bar(range(len(detection)), detection.values, color=bar_colors)
ax_detect.set_xticks(range(len(detection)), detection.index, rotation=90)
ax_detect.set(ylabel="Fraction of records with the cell", ylim=(0, 1))
ax_detect.grid(axis="x", visible=False)

counts = cells_per_record.value_counts().reindex(range(1, len(cells) + 1), fill_value=0)
ax_count.bar(counts.index, counts.values / len(mdt), color=COLORS[0])
ax_count.set(xlabel="Cells in a record", ylabel="Fraction of records")
ax_count.set_xticks(counts.index)
ax_count.grid(axis="x", visible=False)

ax_cdf.plot(*ecdf(received), label="All received cells")
ax_cdf.plot(*ecdf(strongest_rsrp), linestyle="--", label="Strongest cell")
ax_cdf.axvline(cfg.measurement.min_rsrp_dbm, color="gray", linestyle=":", label="Threshold")
ax_cdf.set(xlabel="RSRP (dBm)", ylabel="Empirical CDF", ylim=(0, 1))
ax_cdf.legend(loc="upper left")
label_panels([ax_detect, ax_count, ax_cdf])
save_figure(fig, "scenario_rsrp_statistics")
plt.show()

**Fig. 7.** RSRP statistics. (a) Fraction of the records that contain each cell. The color
identifies the site. (b) Distribution of the number of cells in a record. (c) Empirical
cumulative distribution function of the reported RSRP for all received cells and for the
strongest cell of each record. The dotted line is `measurement.min_rsrp_dbm`.

Table 11 gives the statistics of the reported RSRP for each cell. Table 12 gives the number
of cells and sites that a record contains.

In [ ]:
per_cell = rsrp.describe().T.rename(lambda name: name.removeprefix("rsrp_"))
per_cell = per_cell.rename(
    columns={"count": "Records", "mean": "Mean", "std": "Std", "min": "Min", "max": "Max"}
)
display(
    per_cell.style.set_caption("Table 11. Reported RSRP for each cell (dBm).").format(
        {"Records": "{:.0f}"}, precision=1
    )
)

sites_per_record = pd.DataFrame(
    {
        site: rsrp[[f"rsrp_{c.name}" for c in cells if c.site == site]].notna().any(axis=1)
        for site in sites.index
    }
).sum(axis=1)
pd.Series(
    {
        "Mean number of cells in a record": cells_per_record.mean(),
        "Mean number of sites in a record": sites_per_record.mean(),
        "Fraction of records with one site only": (sites_per_record == 1).mean(),
        "Fraction of records with all sites": (sites_per_record == len(sites)).mean(),
        "Median RSRP of the strongest cell (dBm)": strongest_rsrp.median(),
        "Median RSRP of all received cells (dBm)": np.median(received),
    }
).round(3).to_frame("Value").style.set_caption("Table 12. Cells and sites in a record.")

Fig. 8 shows the strongest cell of each record and the RSRP of that cell against the
three-dimensional distance $d$ to the cell. The reference line is the free-space path loss
for isotropic antennas at the carrier frequency $f$,

$$\mathrm{RSRP}_{\mathrm{FS}}(d) = P_{\mathrm{RS}} - 20 \log_{10} \frac{4 \pi d f}{c}.$$

It is a reference only. The array gain increases the RSRP. Obstructed paths, the antenna
pattern, and the mean of the measurement error decrease it.

In [ ]:
strongest = rsrp.idxmax(axis=1).str.removeprefix("rsrp_")
cell_xyz = cells_frame.set_index("cell")[["x", "y", "z"]]
distance = np.linalg.norm(positions - cell_xyz.loc[strongest].to_numpy(), axis=1)

fig, (ax_map, ax_distance) = plt.subplots(
    1, 2, figsize=(FIGURE_WIDTH_DOUBLE, 3.3), width_ratios=[1.25, 1]
)
ax_map.imshow(is_building, origin="lower", extent=extent, cmap="Greys", alpha=0.3)
for cell in cells:
    chosen = (strongest == cell.name).to_numpy()
    style = {
        "s": 7,
        "color": COLORS[site_order.index(cell.site)],
        "marker": sector_markers[sector_of_cell[cell.name]],
        "linewidths": 0,
    }
    ax_map.scatter(mdt["x"][chosen], mdt["y"][chosen], **style)
    ax_distance.scatter(distance[chosen], strongest_rsrp[chosen], alpha=0.7, **style)
draw_sites(ax_map)
format_map(ax_map)

site_handles = [
    Line2D([], [], linestyle="", marker="o", color=COLORS[index], label=f"Site {site}")
    for index, site in enumerate(site_order)
]
azimuth_handles = [
    Line2D(
        [],
        [],
        linestyle="",
        marker=sector_markers[sector],
        color="gray",
        label=f"Azimuth {azimuth:g}\N{DEGREE SIGN}",
    )
    for sector, azimuth in enumerate(cfg.sites.sector_azimuths_deg)
]
fig.legend(
    handles=site_handles + azimuth_handles,
    loc="outside lower center",
    ncols=len(site_handles) + len(azimuth_handles),
    columnspacing=1.0,
    handletextpad=0.2,
)

reference = np.geomspace(distance.min(), distance.max(), 200)
free_space = cells_frame["power_rs_dbm"].max() - 20 * np.log10(
    4 * np.pi * reference * cfg.simulation.radio.frequency_hz / speed_of_light
)
ax_distance.plot(reference, free_space, color="black", label="Free space, isotropic")
ax_distance.set(xlabel="Distance to the strongest cell $d$ (m)", ylabel="RSRP (dBm)", xscale="log")
ax_distance.legend(loc="lower left", frameon=True, framealpha=0.9, edgecolor="none")
label_panels([ax_map, ax_distance])
save_figure(fig, "scenario_strongest_cell")
plt.show()

**Fig. 8.** Strongest cell. (a) Reported position of each record. The color identifies the
site and the marker identifies the azimuth of the strongest cell. (b) Reported RSRP of the
strongest cell against the distance to that cell, with the free-space reference.

## 5. Verification

The checks below compare the files with the rules of the generator. A check that fails stops
the notebook with an `AssertionError`.

In [ ]:
step = cfg.measurement.rsrp_resolution_db
end = start + pd.Timedelta(days=cfg.traffic.num_days)
since_start = (mdt["timestamp"] - start) % pd.Timedelta(minutes=cfg.traffic.slot_minutes)
azimuths = sorted(cfg.sites.sector_azimuths_deg)
height_min = cfg.ue_height.min_m - 1e-6
height_max = cfg.ue_height.max_m + 1e-6

checks = {
    "Each site has one cell for each azimuth": all(
        sorted(group) == azimuths for _, group in cells_frame.groupby("site")["azimuth_deg"]
    ),
    "Each cell name is unique": cells_frame["cell"].is_unique,
    "Each cell is at the configured height above the ground": np.allclose(
        cells_frame["z"] - height_map.z_ground, cfg.sites.mast_height_m
    ),
    "Each site is in an outdoor grid tile": not is_building[
        grid_index(sites["x"], sites["y"])
    ].any(),
    "Each cell has the configured power": np.allclose(
        cells_frame["power_rs_dbm"], cfg.antenna.power_rs
    ),
    "Each record has a minimum of one cell": bool((cells_per_record >= 1).all()),
    "Each RSRP is a multiple of the resolution": np.allclose(
        received / step, np.round(received / step)
    ),
    "No RSRP is below min_rsrp_dbm": bool((received >= cfg.measurement.min_rsrp_dbm).all()),
    "Each timestamp is the start of a slot": bool((since_start == pd.Timedelta(0)).all()),
    "Each timestamp is in the configured days": bool(
        mdt["timestamp"].between(start, end, inclusive="left").all()
    ),
    "The timestamps are in the order of the time": mdt["timestamp"].is_monotonic_increasing,
    "Each UE height is in the configured range": bool(
        ue_height.between(height_min, height_max).all()
    ),
}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
assert all(checks.values())